In [ ]:
import io
import os
import random
import torch
import json
import pickle
import contextlib
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from ultralytics import YOLO 
from mylib import simsettings
from mylib import myutils
from mylib import simtools
from mylib import dqn
from mylib import yolo_patch_softmax as _
from constants import OBS_SCALE, CELL_SIDE, MAP_RESOLUTION, DISPLAY_STEP
from constants import AGENT_HEIGHT, AGENT_RADIUS 
from constants import MAX_ITER_COEF, CONFIDENCE_THRESHOLD, LOCATION_ERROR_THRESHOLD, PSEUDO_COUNT_THRESHOLD
from constants import ACTIONS      
from dotenv import load_dotenv
load_dotenv()

import habitat_sim
import habitat_sim.nav as nav
from habitat.utils.visualizations import maps
from habitat_sim.utils import common as utils

# Reload imported modules
%load_ext autoreload
%autoreload 2

# Load YOLO model
yolo_model = YOLO("yolo11x.pt")  

# Initialize cuda:0 device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
with contextlib.redirect_stdout(io.StringIO()):
    yolo_model = yolo_model.to(device)

In [ ]:
# Load the JSON file for simulation
with open('simulation-data/simulations.json', 'r') as f:
    simulations = json.load(f)

# Simulation configuration (index selects the simulation)
simulation = simulations[35]

# Access its fields
SCENE = simulation["scene"]
TARGET_OBJECT = simulation["target_object"]
TARGET_OBJECT_ID = simulation["target_object_id"]
REAL_TARGET_LOCATION = simulation["target_object_location"]
INDEX = simulation["index"]

# Load the JSON file for RGB camera intrinsics
with open('simulation-data/camera-intrinsics.json', 'r') as f:
    intrinsics = json.load(f)

# Load bins per class
with open('simulation-data/object-classes-bins.json', 'r') as f:
    classes_bins = json.load(f)

# Load indoor objects
with open('simulation-data/indoor-objects.json', 'r') as f:
    data = json.load(f)
    indoor_objects = [list(item.values())[0] for item in data["indoor_classes"]]

# Load Dirichlet priors
with open('simulation-data/dirichlet-alpha-priors-augmented.pkl', 'rb') as f:
    dirichlet_priors = pickle.load(f)

# Load the JSON file for RGB camera intrinsics
with open('simulation-data/camera-intrinsics.json', 'r') as f:
    intrinsics = json.load(f)

# Simulator configuration
dataset_config_file = os.path.join(os.getenv("AI2THOR_DATA"), "ai2thor-hab.scene_dataset_config.json")
sim_settings = {
    "seed": 1,
    "dataset": dataset_config_file,  # Scene dataset
    "scene": SCENE,  # Scene path
    "width": 1024,  # Spatial resolution of the observations
    "height": int(1024*OBS_SCALE),
    "default_agent": 0,
    "sensor_height": AGENT_HEIGHT,  # Height of sensors in meters
    "color_sensor": True,  # RGB sensor
    "depth_sensor": True,  # Depth sensor
    "enable_physics": False,  # kinematics only
}
# Initialize the simulator
cfg = simsettings.make_cfg(sim_settings)
sim = habitat_sim.Simulator(cfg)

# Initialize agent
agent = sim.initialize_agent(sim_settings["default_agent"])

In [ ]:
class ObjectSearchEnv:
    def __init__(self, simulation, agent, yolo_model):
        self.sim = simulation
        self.yolo_model = yolo_model
        self.agent = agent
        self.actions = list(ACTIONS)

        self._setup_maps()

    def _setup_maps(self):
        navmesh_settings = habitat_sim.NavMeshSettings()
        navmesh_settings.agent_height = AGENT_HEIGHT
        navmesh_settings.agent_radius = AGENT_RADIUS
        navmesh_settings.agent_max_climb = 0.2
        navmesh_settings.agent_max_slope = 45.0
        navmesh_settings.include_static_objects = True
        self.sim.recompute_navmesh(self.sim.pathfinder, navmesh_settings)

        self.topdown_map = maps.get_topdown_map(self.sim.pathfinder, height=0, meters_per_pixel=MAP_RESOLUTION, draw_border=True)
        self.topdown_map = myutils.retain_largest_white_chunk(myutils.add_axis_to_map(myutils.map_to_rgb(self.topdown_map)))
        self.topdown_resolution = self.topdown_map.shape[:2]

        self.grid_map = maps.get_topdown_map(self.sim.pathfinder, height=0, meters_per_pixel=CELL_SIDE, draw_border=False)
        self.grid_map = myutils.retain_largest_white_chunk(myutils.map_to_rgb(self.grid_map))
        self.grid_resolution = self.grid_map.shape[:2]

        self.grid_free_cells, self.world_free_coords, self.map_free_cells = [], [], []

        for x_g in range(self.grid_resolution[0]):
            for y_g in range(self.grid_resolution[1]):
                if self.grid_map[x_g, y_g, 0] == 255:
                    real_wrld_z, real_wrld_x = maps.from_grid(x_g, y_g, self.grid_resolution, pathfinder=self.sim.pathfinder)
                    map_x, map_y = maps.to_grid(real_wrld_z, real_wrld_x, self.topdown_resolution, pathfinder=self.sim.pathfinder)

                    if self.sim.pathfinder.is_navigable([real_wrld_x, 0.0, real_wrld_z]) and self.topdown_map[map_x, map_y, 0] == 255:
                        self.grid_free_cells.append([x_g, y_g])
                        self.world_free_coords.append([real_wrld_x, 0.0, real_wrld_z])
                        self.map_free_cells.append([map_x, map_y])
                    else:
                        self.grid_map[x_g, y_g, :] = [128, 128, 128]

        self.num_free_cells = len(self.grid_free_cells)

    def reset(self):
        self.num_actions = 0
        self.traversed_distance = 0.0
        self.target_found = False

        self.grid_position = random.choice(self.grid_free_cells)
        idx = self.grid_free_cells.index(self.grid_position)
        self.world_position = self.world_free_coords[idx]
        self.map_position = self.map_free_cells[idx]
        self.agent_yaw = random.choice([0, 90, 180, 270])

        agent_state = habitat_sim.AgentState()
        agent_state.position = self.world_position
        agent_state.rotation = myutils.yaw_to_quaternion(self.agent_yaw)
        self.agent.set_state(agent_state)

        self.agent_state = agent_state  # for step()

        min_bounds, max_bounds = self.sim.pathfinder.get_bounds()
        x_dim = max_bounds[0] - min_bounds[0]
        self.topdown_radius = (AGENT_RADIUS / x_dim * self.topdown_resolution[0])
        self.grid_radius = (AGENT_RADIUS / x_dim * self.grid_resolution[0])

        return self._get_state()

    def step(self, action):

        if isinstance(action, int):
            action = self.actions[action]

        if not simtools.is_action_valid(action, self.grid_position, self.agent_yaw, self.grid_free_cells):
            return self._get_state(), -0.1, False, {}
        
        self.grid_position, self.agent_yaw = simtools.perform_action(action, self.grid_position, self.agent_yaw)
        idx = self.grid_free_cells.index(self.grid_position)
        self.world_position = self.world_free_coords[idx]
        self.map_position = self.map_free_cells[idx]

        self.agent_state.position = self.world_position
        self.agent_state.rotation = myutils.yaw_to_quaternion(self.agent_yaw)
        self.agent.set_state(self.agent_state)

        self.num_actions += 1

        obs = self.sim.get_sensor_observations(0)
        rgb = obs["color_sensor"]
        depth = obs["depth_sensor"]
        detections = self._detect_objects(rgb)
        simtools.merge_rgb_yolo_outputs(rgb, detections)
        self.target_found, _ = simtools.was_target_found(TARGET_OBJECT_ID, detections, CONFIDENCE_THRESHOLD)

        # print(f"Action: {action}, Position: {self.grid_position}, Yaw: {self.agent_yaw}, Target Found: {self.target_found}")
        # simtools.display_sim_observations(rgb, depth)
        # simtools.display_topdown_maps(self.topdown_map, self.grid_map, (self.map_position, self.grid_position), (self.topdown_radius, self.grid_radius), self.agent_yaw)
        
        reward = 1.0 if self.target_found else -0.01
        done = self.target_found or self.num_actions >= MAX_ITER_COEF* self.num_free_cells
        return self._get_state(), reward, done, {}

    def _detect_objects(self, rgb):
        results = self.yolo_model.predict(source=rgb[:, :, :3], device='cuda:0', conf=0.30, iou=0.40, verbose=False, max_det=10)
        return simtools.parse_yolo_detections(results)


    def _get_state(self):
        # --- Agent pose (grid-based) ---
        grid_x, grid_z = self.grid_position
        norm_x = grid_x / self.grid_resolution[0]
        norm_z = grid_z / self.grid_resolution[1]
        norm_yaw = self.agent_yaw / 360.0  # normalize to [0,1]

        pose_vec = np.array([norm_x, norm_z, norm_yaw], dtype=np.float32)

        # --- Local occupancy patch (9x9) ---
        patch = np.zeros((9, 9), dtype=np.float32)
        cx, cz = self.grid_position

        for dx in range(-2, 3):
            for dz in range(-2, 3):
                gx = cx + dx
                gz = cz + dz
                if 0 <= gx < self.grid_resolution[0] and 0 <= gz < self.grid_resolution[1]:
                    is_free = self.grid_map[gx, gz, 0] == 255  # white pixel means free
                    patch[dx + 2, dz + 2] = 1.0 if is_free else 0.0

        patch_vec = patch.flatten()

        # --- Goal embedding (index) ---
        goal_idx = float(TARGET_OBJECT_ID)  # just pass as scalar; use embedding in model

        # --- Concatenate all ---
        state_vec = np.concatenate([pose_vec, patch_vec, [goal_idx]], axis=0)

        return state_vec

In [ ]:
# Create the environment
env = ObjectSearchEnv(sim, agent, yolo_model)

In [ ]:
# Hyperparameters
gamma = 0.99
epsilon_start = 1.0
epsilon_end = 0.1
epsilon_decay = 30
batch_size = 64
lr = 1e-3
target_update_freq = 10
max_episodes = 50

load_pretrained = True  # Set to False to train from scratch

# Environment setup
state_dim = len(env.reset())
n_actions = len(env.actions)

# Initialize DQN + target network
policy_net = dqn.DQN(input_dim=state_dim, output_dim=n_actions)
target_net = dqn.DQN(input_dim=state_dim, output_dim=n_actions)

# Load from file if exists
if load_pretrained and os.path.exists("dqn_policy.pth"):
    policy_net.load_state_dict(torch.load("dqn_policy.pth"))

# 
target_net.load_state_dict(policy_net.state_dict())
target_net.eval() # Used for evaluation, no gradients
optimizer = torch.optim.Adam(policy_net.parameters(), lr=lr)
replay_buffer = dqn.ReplayBuffer(capacity=10000)

# Epsilon schedule
def get_epsilon(episode):
    return epsilon_end + (epsilon_start - epsilon_end) * np.exp(-episode / epsilon_decay)



# Training loop
for episode in range(max_episodes):
    state = env.reset()
    step_count = 0  # Track episode length
    episode_reward = 0
    done = False

    while not done:
        epsilon = get_epsilon(episode)
        action = dqn.select_action(state, policy_net, epsilon, n_actions)

        next_state, reward, done, _ = env.step(action)
        replay_buffer.push(state, action, reward, next_state, done)

        state = next_state
        episode_reward += reward
        step_count += 1

        if len(replay_buffer) >= batch_size:
            # Sample and train
            states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

            q_values = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            next_q_values = target_net(next_states).max(1)[0]
            targets = rewards + gamma * next_q_values * (1 - dones)

            loss = F.mse_loss(q_values, targets.detach())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Update target network
    if episode % target_update_freq == 0:
        target_net.load_state_dict(policy_net.state_dict())

    reward_history.append(episode_reward)
    episode_lengths.append(step_count)
    success_flags.append(env.target_found)
    print(f"Episode {episode} | Reward: {episode_reward:.2f} | Steps: {step_count} | Epsilon: {epsilon:.3f}")

# After training, save the model
torch.save(policy_net.state_dict(), "dqn_policy.pth")

# # Evaluation loop
# state = env.reset()
# done = False
# total_reward = 0

# while not done:
#     with torch.no_grad():
#         state_tensor = torch.from_numpy(state).unsqueeze(0).float()
#         action = policy_net(state_tensor).argmax().item()
#     state, reward, done, _ = env.step(action)
#     total_reward += reward

# print("Eval reward:", total_reward)

# Plot reward
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(reward_history)
plt.title("Episode Rewards")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.grid()

# Plot episode lengths
plt.subplot(1, 2, 2)
plt.plot(episode_lengths)
plt.title("Episode Lengths")
plt.xlabel("Episode")
plt.ylabel("Steps")
plt.grid()
plt.tight_layout()
plt.show()

# Print success rate
success_rate = sum(success_flags) / len(success_flags)
print(f"Success Rate: {success_rate:.2f}")
